# Information Retrieval (IR) Laboratory — Practicals 1 to 6
**BE AI & DS | Computer Laboratory II (IR)**

This notebook is organized according to the uploaded IR Lab Manual (2026–27). Each practical has its own heading, objective, dataset notes, commented code, and a small verification/output section.

## Before the exam
1. Run the **Setup** cell once.
2. Run practicals individually, from top to bottom.
3. Keep the required CSV files in the same folder as this notebook when you have them. Each practical includes a small fallback dataset so that you can demonstrate the algorithm if the original dataset is unavailable.
4. The fallback data is for demonstration only; it is not a replacement for the manual's full datasets.

**Manual assignment mapping:** 1 Text preprocessing; 2 Inverted files; 3 Bayesian network for heart disease; 4 Email spam filtering with KNN; 5 Agglomerative hierarchical clustering; 6 PageRank.

## 0. Common setup

Install missing packages in Jupyter with `%pip install ...`. The notebook uses NLTK, pandas, NumPy, Matplotlib, and scikit-learn. `pgmpy` is optional; Practical 3 includes a NumPy-based Bayesian-network demonstration so it can still run when `pgmpy` is not installed.

In [ ]:
# Uncomment this line only if a required package is missing:
# %pip install nltk pandas numpy matplotlib scikit-learn scipy

import os
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Common libraries imported successfully.")

# Practical 1 — Text preprocessing (stop-word removal and stemming)

**Problem statement:** Write a program for pre-processing a text document such as stop-word removal and stemming.

**Objective:** Understand information retrieval and NLP preprocessing.

**Dataset/input:** The manual's Practical 1 is text-document based; it does not prescribe a separate CSV dataset. This cell uses a short sample document and NLTK's Brown Corpus to verify the NLTK installation. Download `brown` and `stopwords` only if you want to use those NLTK resources.

**What this code does:** lowercases text, tokenizes words, removes punctuation and stop words, applies Porter stemming, and optionally demonstrates Brown Corpus access.

In [ ]:
# Step 1: Import and download the NLTK resources required for this practical.
import nltk

# Download only the resources used here. The first download needs internet access.
for resource in ["punkt", "punkt_tab", "stopwords", "brown"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Step 2: Input document (replace this string with text from your own .txt file if required).
document = """
Information Retrieval is the process of finding relevant information from a large
collection of documents. Text preprocessing removes noise and improves searching.
The students are learning retrieval, indexing, searching and ranking documents.
"""

# Step 3: Normalize text (lowercase) and tokenize it into words.
clean_text = document.lower()
try:
    tokens = word_tokenize(clean_text)
except LookupError:
    # Fallback tokenizer if the NLTK punkt resource is unavailable.
    tokens = re.findall(r"\b[a-zA-Z]+\b", clean_text)

# Step 4: Remove punctuation and non-alphabetic tokens.
tokens = [word for word in tokens if word.isalpha()]

# Step 5: Remove English stop words (common words such as 'is', 'the', 'and').
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    # Small fallback list; full NLTK stopwords are preferred.
    stop_words = {"is", "the", "a", "an", "and", "are", "from", "of", "to", "in", "it"}

tokens_without_stopwords = [word for word in tokens if word not in stop_words]

# Step 6: Apply Porter stemming (e.g. learning -> learn, documents -> document).
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in tokens_without_stopwords]

print("Original document:\n", document)
print("\nTokens:", tokens)
print("\nAfter stop-word removal:", tokens_without_stopwords)
print("\nAfter stemming:", stemmed_words)

# Step 7: Verify that NLTK's Brown Corpus is available.
try:
    from nltk.corpus import brown
    print("\nBrown Corpus sample:", brown.words()[:15])
    print("Number of Brown Corpus words:", len(brown.words()))
except LookupError:
    print("\nBrown corpus is not available. Run nltk.download('brown').")

### Practical 1 — Key points
- **Tokenization:** splitting text into words/tokens.
- **Stop-word removal:** removing frequent low-information words.
- **Stemming:** reducing words to a common stem; Porter stemmer is used here.
- **Brown Corpus:** a balanced corpus of American English text, commonly used in NLP research; download with `nltk.download('brown')`.

# Practical 2 — Document retrieval using Inverted Files

**Problem statement:** Implement a program for retrieval of documents using inverted files.

**Objective:** Study indexing and searching with an inverted index.

**Dataset/input:** The manual demonstrates reading `file.txt` line by line. No external dataset is mandatory. This notebook creates three sample documents; you can replace them with your own `.txt` files.

**What this code does:** reads documents, tokenizes words, builds a term-to-document posting list, and retrieves documents containing a query term.

In [ ]:
# Step 1: Create a small document collection.
# Replace these sample strings with text loaded from separate .txt files if available.
documents = {
    "doc1": "Information retrieval finds relevant documents from a collection.",
    "doc2": "An inverted index stores each word and the documents containing it.",
    "doc3": "Search engines use indexing and ranking to retrieve web pages."
}

# Step 2: Define a simple tokenizer (lowercase alphabetic words).
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

# Step 3: Build the inverted index.
# Key = term; Value = list of document IDs in which the term occurs.
inverted_index = {}

for doc_id, text in documents.items():
    terms = set(tokenize(text))  # set avoids duplicate doc IDs for repeated terms
    for term in terms:
        inverted_index.setdefault(term, []).append(doc_id)

print("INVERTED INDEX")
for term in sorted(inverted_index):
    print(f"{term}: {sorted(inverted_index[term])}")

# Step 4: Search the index using a query.
def search(query):
    query_terms = tokenize(query)
    if not query_terms:
        return []
    # AND search: return documents containing every query term.
    result_sets = [set(inverted_index.get(term, [])) for term in query_terms]
    matching_docs = set.intersection(*result_sets) if result_sets else set()
    return sorted(matching_docs)

query = "inverted index"
print("\nQuery:", query)
print("Retrieved documents:", search(query))

# Step 5: Display the actual retrieved document text.
for doc_id in search(query):
    print(doc_id, "->", documents[doc_id])

### Practical 2 — Key points
- **Inverted index:** maps each term to the list of documents containing that term.
- **Posting list:** the document IDs stored against a term.
- **Why it is useful:** search does not need to scan every document for every query.
- The search function above uses AND semantics for multiple query terms.

# Practical 3 — Bayesian Network for Heart Disease Prediction

**Problem statement:** Construct a Bayesian network considering medical data and demonstrate diagnosis using the standard Heart Disease dataset (the manual mentions the Cleveland Heart Disease dataset).

**Objective:** Evaluate retrieved information and study a Bayesian network model.

**Dataset expected:** `heart.csv` (or the equivalent Cleveland heart-disease CSV). Put it in the same folder as this notebook. The commonly used Cleveland data has attributes such as `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, `thal`, and target `target`/`num`.

**Important:** The manual's printed example uses the Cleveland dataset and `pgmpy`. The code below provides a compact, runnable Bayesian-network example using a small, clearly labelled demonstration dataset if `heart.csv` is not present. It is a teaching demonstration, not a clinical prediction tool.

In [ ]:
# Step 1: Load the heart-disease dataset if supplied.
# Expected file: heart.csv in the notebook's current working directory.
if os.path.exists("heart.csv"):
    heart_df = pd.read_csv("heart.csv")
    print("Loaded heart.csv:", heart_df.shape)
else:
    print("heart.csv not found. Using a tiny DEMONSTRATION dataset.")
    # Demo rows only; use the actual Cleveland/heart.csv dataset for the exam if provided.
    heart_df = pd.DataFrame({
        "age":       [63, 67, 41, 62, 60, 55, 58, 50],
    })

In [ ]:
# Rebuild the fallback dataset completely if heart.csv is unavailable.
if not os.path.exists("heart.csv"):
    heart_df = pd.DataFrame({
        "age": [63, 67, 41, 62, 60, 55, 58, 50, 45, 70, 52, 48],
        "cp": [3, 4, 2, 4, 1, 2, 3, 2, 1, 4, 3, 1],
        "thalach": [150, 108, 172, 160, 132, 150, 140, 165, 178, 110, 155, 170],
        "exang": [0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0],
        "target": [0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0]
    })

# Standardize the target column name for common Cleveland CSV variants.
if "target" not in heart_df.columns:
    if "num" in heart_df.columns:
        heart_df = heart_df.rename(columns={"num": "target"})
    elif "HeartDisease" in heart_df.columns:
        heart_df = heart_df.rename(columns={"HeartDisease": "target"})

# Convert Cleveland target values 1-4 into binary disease/no-disease labels.
heart_df = heart_df.copy()
heart_df["target"] = pd.to_numeric(heart_df["target"], errors="coerce").fillna(0)
heart_df["target"] = (heart_df["target"] > 0).astype(int)

# Select attributes that are commonly present in the Cleveland dataset.
required = ["age", "cp", "thalach", "exang", "target"]
missing = [c for c in required if c not in heart_df.columns]
if missing:
    raise ValueError(f"heart.csv is missing required columns: {missing}")

heart_df = heart_df[required].dropna().copy()

# Discretize continuous variables to create categorical Bayesian-network nodes.
heart_df["age_group"] = pd.cut(heart_df["age"], bins=[0, 45, 60, 120],
                               labels=["young", "middle", "senior"], include_lowest=True).astype(str)
heart_df["hr_group"] = pd.cut(heart_df["thalach"], bins=[0, 120, 160, 300],
                              labels=["low", "medium", "high"], include_lowest=True).astype(str)
heart_df["cp"] = heart_df["cp"].astype(str)
heart_df["exang"] = heart_df["exang"].astype(str)
heart_df["target"] = heart_df["target"].astype(str)

# Naive Bayes is a simple Bayesian-network structure:
# age_group, cp, hr_group, exang -> target.
# Estimate P(target) and P(feature value | target) with Laplace smoothing.
features = ["age_group", "cp", "hr_group", "exang"]
classes = sorted(heart_df["target"].unique())

def predict_heart_disease(row):
    scores = {}
    for cls in classes:
        subset = heart_df[heart_df["target"] == cls]
        # Prior probability with smoothing.
        log_prob = np.log((len(subset) + 1) / (len(heart_df) + len(classes)))
        for feature in features:
            value = str(row[feature])
            value_count = (subset[feature].astype(str) == value).sum()
            # Laplace smoothing avoids zero probabilities.
            k = heart_df[feature].nunique()
            conditional = (value_count + 1) / (len(subset) + max(k, 1))
            log_prob += np.log(conditional)
        scores[cls] = log_prob
    return max(scores, key=scores.get), scores

# Example patient: values are illustrative categories, not medical advice.
sample_patient = {"age_group": "senior", "cp": "4", "hr_group": "low", "exang": "1"}
predicted_class, log_scores = predict_heart_disease(sample_patient)

print("Bayesian Network structure: age_group, cp, hr_group, exang -> target")
print("Training rows:", len(heart_df))
print("Example patient:", sample_patient)
print("Predicted target (0 = no disease, 1 = disease):", predicted_class)
print("Log probability scores:", log_scores)

# Optional: show the pgmpy imports used in many lab-manual implementations.
try:
    import pgmpy
    print("pgmpy is installed. Version:", getattr(pgmpy, "__version__", "unknown"))
except ImportError:
    print("pgmpy is not installed. This notebook uses the equivalent simple Bayesian-network calculation above.")

### Practical 3 — Dataset and interpretation
- Use the **Cleveland Heart Disease dataset** supplied by your lab/teacher, commonly saved as `heart.csv`.
- The manual lists clinical attributes including age, sex, chest-pain type (`cp`), resting blood pressure, cholesterol, maximum heart rate (`thalach`), exercise-induced angina (`exang`), oldpeak, slope, ca, thal, and heart-disease class.
- The code discretizes selected attributes and estimates conditional probabilities with Laplace smoothing.
- `0` means no disease and `1` means disease in this binary demonstration. Do not interpret the result as real medical advice.

# Practical 4 — Email Spam Filtering using K-Nearest Neighbours (KNN)

**Problem statement:** Implement email spam filtering using a text-classification algorithm and an appropriate dataset.

**Objective:** Understand spam filtering and the KNN algorithm.

**Dataset mentioned in the manual:** Enron email corpus (the manual refers to Enron ham/spam folders). A commonly used alternative is a labelled `spam.csv` with columns `v1` and `v2` (or `label` and `message`).

**What this code does:** loads `spam.csv` if present; otherwise it uses a small demonstration set. It converts email text to TF-IDF vectors, trains KNN, predicts a test set, and prints accuracy and a confusion matrix.

In [ ]:
# Step 1: Load a labelled email dataset.
# Supported CSV formats:
#   A) spam.csv with columns v1=ham/spam and v2=email text
#   B) spam.csv with columns label and message
if os.path.exists("spam.csv"):
    spam_df = pd.read_csv("spam.csv", encoding="latin-1")
    if {"v1", "v2"}.issubset(spam_df.columns):
        spam_df = spam_df.rename(columns={"v1": "label", "v2": "message"})
    elif not {"label", "message"}.issubset(spam_df.columns):
        raise ValueError("spam.csv must contain v1/v2 or label/message columns.")
    spam_df = spam_df[["label", "message"]].dropna()
    print("Loaded spam.csv:", spam_df.shape)
else:
    print("spam.csv not found. Using a small DEMONSTRATION email dataset.")
    spam_df = pd.DataFrame({
        "label": ["ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam",
                  "ham", "spam", "ham", "spam"],
        "message": [
            "Are we meeting for the project today",
            "Congratulations you have won a free prize claim now",
            "Please send me the class notes",
            "Urgent claim your cash reward by clicking this link",
            "I will call you after the lecture",
            "You are selected for a free gift card",
            "Can you share the assignment pdf",
            "Win money now limited offer click here",
            "The practical exam is tomorrow morning",
            "Get free entry in our prize competition",
            "Thanks for helping with the notes",
            "Claim your exclusive offer now"
        ]
    })

# Step 2: Convert labels to consistent lowercase strings.
spam_df["label"] = spam_df["label"].astype(str).str.lower().str.strip()
spam_df["label"] = spam_df["label"].replace({"not spam": "ham", "legitimate": "ham"})
spam_df = spam_df[spam_df["label"].isin(["ham", "spam"])].copy()

# Step 3: Split text and labels into training and testing data.
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train_text, X_test_text, y_train, y_test = train_test_split(
    spam_df["message"].astype(str), spam_df["label"],
    test_size=0.25, random_state=42, stratify=spam_df["label"]
)

# Step 4: Convert email text into numerical TF-IDF feature vectors.
vectorizer = TfidfVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Step 5: Train the KNN classifier.
# K is kept small for a small lab dataset; tune it for a larger corpus.
k = min(3, len(y_train))
knn_model = KNeighborsClassifier(n_neighbors=max(1, k), metric="cosine", algorithm="brute")
knn_model.fit(X_train, y_train)

# Step 6: Predict and evaluate model performance.
y_pred = knn_model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:\n", classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix [ham, spam]:\n", confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))

# Step 7: Test a new email.
new_email = ["Congratulations! Claim your free cash prize now"]
new_vector = vectorizer.transform(new_email)
print("\nNew email:", new_email[0])
print("Prediction:", knn_model.predict(new_vector)[0])

### Practical 4 — Dataset notes
- **Enron corpus:** real email messages arranged into ham/spam examples in some prepared versions; the manual refers to Enron corpus directories.
- **Alternative:** the widely used SMS Spam Collection CSV can be adapted, but it contains SMS messages rather than actual emails.
- **TF-IDF:** gives higher weight to terms that are informative in a document and lower weight to terms common across many documents.
- **KNN:** classifies a new message based on the labels of its nearest training examples.

# Practical 5 — Agglomerative Hierarchical Clustering

**Problem statement:** Implement Agglomerative Hierarchical Clustering using an appropriate dataset.

**Dataset named in the manual:** `Credit_Card.csv` / Credit Card dataset. The manual's code removes the `CUST_ID` column, fills missing values, scales the features, applies PCA to two components, plots a dendrogram, and compares cluster counts using silhouette score.

**What this code does:** loads `Credit_Card.csv` if present; otherwise creates a synthetic numeric dataset strictly for demonstrating the workflow. It scales features, reduces dimensions with PCA, plots a dendrogram, fits AgglomerativeClustering for k=2 to 6, and compares silhouette scores.

In [ ]:
# Step 1: Load the Credit Card dataset.
# Keep Credit_Card.csv in the same folder as this notebook.
if os.path.exists("Credit_Card.csv"):
    credit_df = pd.read_csv("Credit_Card.csv")
    print("Loaded Credit_Card.csv:", credit_df.shape)
else:
    print("Credit_Card.csv not found. Creating a SYNTHETIC demonstration dataset.")
    from sklearn.datasets import make_blobs
    synthetic_X, _ = make_blobs(n_samples=300, centers=4, n_features=6,
                                cluster_std=2.0, random_state=42)
    credit_df = pd.DataFrame(synthetic_X, columns=[
        "BALANCE", "PURCHASES", "CASH_ADVANCE", "CREDIT_LIMIT",
        "PAYMENTS", "MINIMUM_PAYMENTS"
    ])

# Step 2: Remove customer ID because it is an identifier, not a clustering feature.
if "CUST_ID" in credit_df.columns:
    credit_df = credit_df.drop(columns=["CUST_ID"])

# Step 3: Keep numeric columns and handle missing values.
X_credit = credit_df.select_dtypes(include=[np.number]).copy()
X_credit = X_credit.replace([np.inf, -np.inf], np.nan)
X_credit = X_credit.fillna(X_credit.median())
if X_credit.shape[1] < 2:
    raise ValueError("The dataset must contain at least two numeric feature columns.")

# Step 4: Standardize features so that large-scale columns do not dominate distance.
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_credit)

# Step 5: Normalize and reduce data to 2 principal components for visualization.
X_normalized = normalize(X_scaled)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_normalized)
print("Explained variance by two PCs:", pca.explained_variance_ratio_.sum())

# Step 6: Draw a dendrogram. Use a sample for readability on large datasets.
sample_for_tree = X_pca[:min(300, len(X_pca))]
linked = linkage(sample_for_tree, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(linked, truncate_mode="lastp", p=25)
plt.title("Dendrogram — Ward linkage")
plt.xlabel("Merged sample / cluster")
plt.ylabel("Euclidean distance")
plt.show()

# Step 7: Fit hierarchical clustering for different values of k.
k_values = [k for k in [2, 3, 4, 5, 6] if k < len(X_pca)]
silhouette_scores = []
cluster_models = {}

for k in k_values:
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(X_pca)
    cluster_models[k] = labels
    score = silhouette_score(X_pca, labels)
    silhouette_scores.append(score)
    print(f"k={k}, silhouette score={score:.4f}")

# Step 8: Plot the clustering result for k=2 and the silhouette comparison.
if 2 in cluster_models:
    plt.figure(figsize=(6, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_models[2], s=18)
    plt.title("Agglomerative clustering (k=2)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.show()

plt.figure(figsize=(7, 4))
plt.bar(k_values, silhouette_scores)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score comparison")
plt.show()

best_k = k_values[int(np.argmax(silhouette_scores))]
print("Best k based on silhouette score:", best_k)

### Practical 5 — Key points
- **Agglomerative clustering:** bottom-up hierarchical clustering; initially every point is its own cluster, then the closest clusters are merged.
- **Ward linkage:** merges clusters to minimize the increase in within-cluster variance.
- **Dendrogram:** tree diagram showing the order and distance of cluster merges.
- **PCA:** reduces many features to two components for visualization.
- **Silhouette score:** higher values generally indicate better-separated clusters.
- For exam use, replace the synthetic data with the actual `Credit_Card.csv` file from your lab/manual source.

# Practical 6 — PageRank Algorithm

**Problem statement:** Implement the PageRank algorithm (using Python or BeautifulSoup, as stated in the manual).

**Objective:** Study PageRank and the Random Walk model.

**Input/dataset:** A directed web graph. The manual illustrates a small graph with five pages (nodes 0–4), discusses transition matrices, random walk, dead ends/spider traps, and teleportation. No external dataset is mandatory; this notebook uses a small directed graph that can be edited.

**What this code does:** builds a directed graph, calculates PageRank using the power-iteration method with damping factor 0.85, handles dangling pages, and prints the ranking. It also compares with NetworkX if installed.

In [ ]:
# Step 1: Define a directed graph.
# An edge (A, B) means page A contains a link to page B.
# Edit these edges to represent another web graph.
nodes = [0, 1, 2, 3, 4]
edges = [
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 0), (2, 3),
    (3, 2), (3, 4),
    (4, 1)
]

# Step 2: Create the row-stochastic transition matrix.
# If a page has no outgoing links (dangling node), treat it as linking to every page.
n = len(nodes)
node_to_index = {node: i for i, node in enumerate(nodes)}
M = np.zeros((n, n), dtype=float)

for source, destination in edges:
    M[node_to_index[source], node_to_index[destination]] += 1

for i in range(n):
    row_sum = M[i].sum()
    if row_sum == 0:
        M[i, :] = 1 / n
    else:
        M[i, :] /= row_sum

print("Transition matrix (rows = source pages, columns = destination pages):")
print(M)

# Step 3: Apply PageRank power iteration.
# d (damping factor) = probability of following a link.
# (1-d)/n is the teleportation probability for each page.
d = 0.85
rank = np.ones(n) / n
tolerance = 1e-10
max_iterations = 100

for iteration in range(max_iterations):
    new_rank = (1 - d) / n + d * (M.T @ rank)
    if np.linalg.norm(new_rank - rank, ord=1) < tolerance:
        rank = new_rank
        break
    rank = new_rank

# Step 4: Display pages in descending PageRank order.
ranking = sorted(zip(nodes, rank), key=lambda item: item[1], reverse=True)
print(f"\nConverged after {iteration + 1} iterations")
print("PageRank scores:")
for page, score in ranking:
    print(f"Page {page}: {score:.6f}")

print("\nSum of PageRank scores:", round(rank.sum(), 6))

# Step 5: Optional comparison with NetworkX implementation.
try:
    import networkx as nx
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    graph.add_edges_from(edges)
    nx_rank = nx.pagerank(graph, alpha=d, max_iter=100, tol=1e-10)
    print("\nNetworkX PageRank:")
    for page, score in sorted(nx_rank.items(), key=lambda item: item[1], reverse=True):
        print(f"Page {page}: {score:.6f}")
except ImportError:
    print("\nNetworkX not installed; custom PageRank result shown above.")

### Practical 6 — Key points
- **PageRank:** assigns importance to a web page based on incoming links and the importance of linking pages.
- **Random surfer:** a user follows links with probability `d`; with probability `1-d`, the user teleports to a random page.
- **Damping factor:** commonly set to 0.85.
- **Dangling node:** a page with no outgoing links; this implementation treats it as linking uniformly to all pages.
- **Power iteration:** repeatedly updates the rank vector until it converges.

# Quick revision — dataset/file checklist

| Practical | Dataset/input | Expected file/resource |
|---|---|---|
| 1 | Sample text + NLTK Brown Corpus (optional verification) | NLTK `brown`, `stopwords` |
| 2 | Text documents | Sample documents in code or `.txt` files |
| 3 | Cleveland Heart Disease | `heart.csv` (or equivalent Cleveland CSV) |
| 4 | Enron ham/spam emails; labelled CSV alternative | Enron folders or `spam.csv` |
| 5 | Credit Card customer data | `Credit_Card.csv` |
| 6 | Directed web graph | Nodes and edges defined in code |

**Exam reminder:** If your examiner gives a specific dataset/file, use that file rather than the demonstration fallback. Always explain the input, preprocessing, algorithm, and output in your own words.